# Лабораторная № 7, вариант 5 «Scrapy-краулер» — анализ прогонов

Источник данных — только файлы прогонов: структурные журналы `logs/run_*.jsonl`, отчёты `reports/run_report_*.json`, сводки `reports/*_summary.json`. Блокнот исполняется после `./run_real.sh` (*Restart & Run All*).

**Две группы прогонов.** Метки без префикса (`A`, `B`, `C`, `S1_cond`, `httpbin_*`, `real_*`) — реальные учебные площадки. Метки `mock_*` — локальная имитация (`tools/mock_shop.py`): на ней воспроизводятся отказы, которые нельзя вызывать на чужом сайте (429 с Retry-After, рост латентности, заглушка-капча, авария домена). Цифры имитации в отчёт идут только с пометкой «(на локальной имитации)».

| Раздел | Содержание |
|---|---|
| 1 | все прогоны: сводная таблица |
| 2 | robots.txt: can_fetch, Crawl-delay, Request-rate, Sitemap |
| 3 | классификация ошибок и журнал попыток (Retry-After vs backoff) |
| 4 | сценарий 429 от домена |
| 5 | срабатывание AutoThrottle (график задержки и латентности) |
| 6 | прерывание и возобновление |
| 7 | предохранитель: изоляция отказа домена |
| 8 | экономия нагрузки, прогоны A/B/C |
| 9 | детектор блокировок и сравнение с историей |
| 10 | отчёт о прогоне, пороги и оповещения |
| 11 | графики: распределение латентности, коды ответа во времени |
| 12 | лимит при 4 процессах |

In [ ]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from src.retry import classification_table
from src.observability import evaluate_alerts, THRESHOLDS

REP, LOGS = ROOT / "reports", ROOT / "logs"

def report(label):
    p = REP / f"run_report_{label}.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None

def events(label):
    files = sorted(LOGS.glob(f"run_{label}_2*.jsonl"))
    if not files:
        return pd.DataFrame()
    df = pd.read_json(files[-1], lines=True)
    df["t"] = (pd.to_datetime(df["ts"]) - pd.to_datetime(df["ts"]).min()).dt.total_seconds()
    return df

def first_existing(*labels):
    for lab in labels:
        if report(lab):
            return lab
    return None

reports = {p.stem.replace("run_report_", ""): json.loads(p.read_text(encoding="utf-8"))
           for p in sorted(REP.glob("run_report_*.json"))}
print("Отчётов о прогонах:", len(reports))
print("Реальные:", [k for k in reports if not reports[k].get("local_mock")])
print("Имитация:", [k for k in reports if reports[k].get("local_mock")])

## 1. Все прогоны

In [ ]:
cols = ["mode", "requests", "error_rate", "share_429_403", "retries", "latency_median_ms", "latency_p95_ms",
        "traffic_kb", "not_modified_304", "incremental_skipped", "breaker_activations", "incidents",
        "items_scraped", "items_dropped", "deferred_urls", "duration_s", "status", "local_mock"]
overview = pd.DataFrame({k: {c: r.get(c) for c in cols} for k, r in reports.items()}).T
overview

## 2. robots.txt

`can_fetch` — итог по более строгому из двух парсеров: `urllib.robotparser` (требование задания) и Protego (RFC 9309, его использует Scrapy). Python ≤ 3.12 не понимает шаблоны `*`/`$`: на правило `Disallow: /ecommerce/*` (так закрыт scrapingcourse.com) он ответил бы «можно».

In [ ]:
rows = []
sources = []
for rr in sorted(REP.glob("robots_report_*.json")):
    sources += json.loads(rr.read_text(encoding="utf-8"))
for r in reports.values():
    sources += r.get("robots") or []
seen = set()
for r in sources:
    key = (r.get("origin"), r.get("status"))
    if key in seen:
        continue
    seen.add(key)
    rows.append({"сайт": r.get("origin"), "HTTP": r.get("status"), "Crawl-delay": r.get("crawl_delay"),
                 "Request-rate": r.get("request_rate"), "Sitemap": ", ".join(r.get("sitemaps") or []) or "—",
                 "can_fetch": "; ".join(f"{u.split('://')[-1]} → {ok}" for u, ok in (r.get("can_fetch") or {}).items()),
                 "трактовка": r.get("policy_note")})
pd.DataFrame(rows)

## 3. Классификация ошибок и журнал попыток

In [ ]:
display(pd.DataFrame(classification_table()))
lab = first_existing("httpbin_codes", "mock_probe_codes")
ev = events(lab) if lab else pd.DataFrame()
print("Журнал попыток прогона:", lab, "(на локальной имитации)" if lab and lab.startswith("mock") else "")
if len(ev):
    keep = ev[ev["event"].isin(["request", "retry", "giveup", "auth_required", "incident", "probe_result"])]
    cols = [c for c in ["t", "event", "url", "status", "attempt", "delay_s", "delay_source", "retry_after_header",
                        "reason", "error_class", "retry_after_parsed_s", "verdict"] if c in keep]
    display(keep[cols].round(2))

## 4. Сценарий варианта: 429 от домена

In [ ]:
for lab in [l for l in ("mock_429", "mock_429_polite") if report(l)]:
    r = report(lab)
    print(f"{lab}: запросов {r['requests']}, коды {r['status_counts']}, повторов {r['retries']} "
          f"(источники задержки {r['retry_sources']}), записей {r['items_scraped']}, {r['duration_s']} с (на локальной имитации)")
ev = events("mock_429")
if len(ev):
    cols = [c for c in ["t", "event", "url", "status", "attempt", "delay_s", "delay_source", "retry_after_header",
                        "pause_s", "source", "adaptive_delay_s"] if c in ev]
    display(ev[ev["event"].isin(["retry", "domain_pause", "giveup"])][cols].round(3))

## 5. Сценарий варианта: срабатывание AutoThrottle

In [ ]:
lab = first_existing("B", "mock_autothrottle")
for lab in [l for l in ("mock_autothrottle", "B", "C") if report(l)]:
    ev = events(lab)
    at = ev[ev["event"] == "autothrottle"]
    if not len(at):
        continue
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(at["t"], at["ms"], ".", label="латентность ответа, мс")
    ax.step(at["t"], at["delay_ms"], where="post", label="задержка AutoThrottle (slot.delay), мс")
    ax.set_xlabel("время от начала прогона, с"); ax.set_ylabel("мс"); ax.legend()
    ax.set_title(f"AutoThrottle: {lab}" + (" (на локальной имитации)" if report(lab).get("local_mock") else ""))
    plt.tight_layout(); plt.show()
    print(lab, "задержка AutoThrottle, мс: мин", at["delay_ms"].min(), "макс", at["delay_ms"].max())

## 6. Сценарий варианта: прерывание и возобновление

In [ ]:
for name in ("real_resume_summary.json", "mock_resume_summary.json"):
    p = REP / name
    if p.exists():
        s = json.loads(p.read_text(encoding="utf-8"))
        print(name, "(на локальной имитации)" if s.get("local_mock") else "")
        display(pd.DataFrame([{"прогон": "1 (SIGINT)", **{k: v for k, v in s["run1"].items() if k != "scrapy_stats"}},
                              {"прогон": "2 (возобновление)", **{k: v for k, v in s["run2"].items() if k != "scrapy_stats"}}]))
        print("Файлы JOBDIR после прерывания:", *s["jobstate_files"], sep="\n  ")
        print("Записей в БД по прогонам:", s["db_rows_by_run"])

## 7. Предохранитель: изоляция отказа домена

In [ ]:
for lab in [l for l in ("real_breaker", "mock_breaker") if report(l)]:
    ev = events(lab)
    print(lab, "(на локальной имитации)" if report(lab).get("local_mock") else "", "| предохранитель:", report(lab)["breakers"])
    display(ev[ev["event"] == "breaker"][["t", "host", "from_state", "to_state", "reason"]].round(1))
    req = ev[ev["event"] == "request"]
    fig, ax = plt.subplots(figsize=(10, 3))
    for i, (host, g) in enumerate(req.groupby("host")):
        ax.scatter(g["t"], [i] * len(g), c=["tab:red" if (s or 0) >= 500 else "tab:green" for s in g["status"]], s=14)
    rej = ev[ev["event"] == "breaker_reject"]
    hosts = list(req.groupby("host").groups)
    ax.scatter(rej["t"], [hosts.index(h) - 0.25 if h in hosts else 0 for h in rej["host"]], marker="x", c="gray", s=12,
               label="не отправлен (open/half-open)")
    ax.set_yticks(range(len(hosts))); ax.set_yticklabels(hosts); ax.set_xlabel("с"); ax.legend(loc="upper right")
    ax.set_title(f"{lab}: запросы по доменам (зелёный — ок, красный — 5xx)")
    plt.tight_layout(); plt.show()

## 8. Экономия нагрузки и прогоны A/B/C

In [ ]:
cmp_md = REP / "comparison.md"
if cmp_md.exists():
    from IPython.display import Markdown
    display(Markdown(cmp_md.read_text(encoding="utf-8")))
abc = [l for l in ("A", "B", "C", "mock_A", "mock_B", "mock_C") if report(l)]
if abc:
    df = pd.DataFrame({l: {"время, с": report(l)["duration_s"], "запросов": report(l)["requests"],
                           "трафик, КБ": report(l)["traffic_kb"], "записей": report(l)["items_scraped"]} for l in abc}).T
    df.plot(kind="bar", subplots=True, layout=(1, 4), figsize=(13, 3), legend=False, rot=0)
    plt.suptitle("Прогоны A / B / C (mock_* — на локальной имитации)"); plt.tight_layout(); plt.show()

## 9. Детектор блокировок и сравнение с историей

In [ ]:
inc = sorted((LOGS / "incidents").glob("*/meta.json"))
rows = [json.loads(p.read_text(encoding="utf-8")) | {"артефакты": str(p.parent.relative_to(ROOT))} for p in inc]
if rows:
    display(pd.DataFrame(rows)[["run_id", "url", "status", "bytes", "verdict", "signals", "артефакты"]])
for lab in [l for l in ("mock_hist_captcha", "B") if report(l)]:
    h = report(lab).get("history") or {}
    print(f"{lab}: статус истории — {h.get('status')}; предыдущих прогонов {h.get('previous_runs')}")
    if h.get("checks"):
        display(pd.DataFrame(h["checks"]))

## 10. Отчёт о прогоне: пороги и оповещения

In [ ]:
display(pd.DataFrame([{"метрика": t.key, "условие": f"{t.op} {t.value}", "уровень": t.severity, "обоснование": t.why}
                      for t in THRESHOLDS]))
rows = []
for lab, r in reports.items():
    for a in r.get("alerts", []):
        rows.append({"прогон": lab, **a})
pd.DataFrame(rows) if rows else "оповещений нет"

## 11. Графики по журналу: распределение латентности и коды ответа во времени

In [ ]:
lab = first_existing("B", "C", "mock_429", "mock_B")
ev = events(lab)
req = ev[(ev["event"] == "request") & ev["ms"].notna()]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
req["ms"].plot(kind="hist", bins=30, ax=axes[0], edgecolor="white")
axes[0].set_title(f"Распределение латентности, мс ({lab})"); axes[0].set_xlabel("мс")
axes[0].axvline(req["ms"].median(), color="k", ls="--", label=f"медиана {req['ms'].median():.0f}")
axes[0].axvline(req["ms"].quantile(0.95), color="r", ls=":", label=f"p95 {req['ms'].quantile(0.95):.0f}")
axes[0].legend()
lab2 = first_existing("mock_429", "B")
ev2 = events(lab2)
r2 = ev2[ev2["event"] == "request"].copy()
r2["сек"] = r2["t"].astype(int)
r2["код"] = r2["status"].fillna(-1).astype(int).astype(str).replace({"-1": "сеть"})
r2.pivot_table(index="сек", columns="код", values="url", aggfunc="count", fill_value=0).plot(
    kind="bar", stacked=True, ax=axes[1], width=0.9)
axes[1].set_title(f"Коды ответа по секундам ({lab2})"); axes[1].set_xlabel("секунда прогона")
axes[1].xaxis.set_major_locator(plt.MaxNLocator(12))
plt.tight_layout(); plt.show()
print("(для меток mock_* — на локальной имитации)")

## 12. Лимит при 4 параллельных процессах

In [ ]:
p = REP / "shared_limit.json"
if p.exists():
    s = json.loads(p.read_text(encoding="utf-8"))
    print("(на локальной имитации)" if s.get("local_mock") else "")
    display(pd.DataFrame(s["rows"]))
else:
    print("Нет reports/shared_limit.json: python3 tools/demo_shared_limit.py")

## Выводы

Заполняются после реального прогона по таблицам выше:

1. Цена вежливости (B против A): ⟪ЗАПОЛНИТЬ⟫ с на одинаковом объёме; A получил ⟪ЗАПОЛНИТЬ⟫ ответов 429/ошибок.
2. Экономия трафика C против B: ⟪ЗАПОЛНИТЬ⟫ %; запросов ⟪ЗАПОЛНИТЬ⟫ → ⟪ЗАПОЛНИТЬ⟫.
3. AutoThrottle на реальном сайте менял задержку в диапазоне ⟪ЗАПОЛНИТЬ⟫ мс.
4. Возобновление: после прерывания повторно не запрошено ⟪ЗАПОЛНИТЬ⟫ URL (dupefilter/filtered), дублей по SKU нет.